# `mpl_figure_saver` — save & reopen matplotlib figures (MATLAB-`.fig`-style)

This notebook demonstrates the essential capabilities and the core algorithm.

**The problem.** A matplotlib `savefig` gives you a *picture*. When you later need
the numbers, a different y-scale, or one more annotation, you re-run the whole
analysis. `mpl_figure_saver` makes every save also drop two sidecar files:

| file | what it is | reopen with | survives matplotlib upgrades? |
|------|------------|-------------|-------------------------------|
| `.png`/`.pdf`/… | the image you asked for | — | — |
| `.mplpkl` | a pickle of the **exact** figure | `loadfig()` | no (pickle) |
| `.mplscene` | the **arrays** behind every artist | `replot_scene()` / `load_arrays()` | **yes** |

So you get a fast, pixel-perfect reopen (pickle) *and* a version-proof fallback
(arrays). Both from one `savefig` or one click of the toolbar **Save** button.


In [ ]:
import matplotlib
# For a real session use:  %matplotlib qt   or   %matplotlib widget
# Here we stay headless so the notebook runs anywhere (CI / SSH).
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

import mpl_figure_saver as mfs

OUT = Path("nb_out"); OUT.mkdir(exist_ok=True)

## 1. Install

One call patches `fig.savefig()` and the Qt / ipympl toolbar **Save** buttons,
and sets configuration. It's safe to re-run (e.g. under `%autoreload`).

- `save_dir` — only required for the widget **`timestamp`** mode (headless/SSH).
- `widget_mode='timestamp'` writes `plot_<timestamp>/` folders; `'dialog'` opens
  a local save dialog (use on a laptop / remote-desktop, not over bare SSH).
- In **VS Code**, call `mfs.fix_vscode_widget_style()` once so `%matplotlib widget`
  figures follow your theme instead of showing a white box.


In [ ]:
mfs.install(save_dir=OUT / "widget_saves", widget_mode="timestamp")
# mfs.fix_vscode_widget_style()   # uncomment inside VS Code

## 2. Save once — get image + arrays + pickle

Nothing special in your plotting code; just call `savefig` as usual.
Note the `cos` line has **no** `label=` and the `axvline` is a reference marker —
both are captured correctly (a bug in naive extractors).

In [ ]:
x = np.linspace(0, 2*np.pi, 200)
fig, ax = plt.subplots()
ax.plot(x, np.sin(x), label="sin")
ax.plot(x, np.cos(x))                 # unlabeled — still captured
ax.axvline(np.pi, color="k", ls="--") # reference line — captured separately
ax.set(xlabel="phase", ylabel="amplitude", title="Trig")
ax.legend()
fig.savefig(OUT / "trig.png")

sorted(p.name for p in OUT.glob("trig.*"))

## 3. Reopen the **exact** figure (pickle)

`loadfig()` returns a live, editable figure with the original layout intact.
It also returns `extra_data` — a free-form dict you can attach at save time
(see §7); empty here because the automatic save path doesn't set it.

In [ ]:
fig2, extra = mfs.loadfig(OUT / "trig.mplpkl")
fig2.axes[0].set_title("Trig (reopened & edited)")
print("extra_data:", extra)
fig2.axes[0].get_title()

## 4. The algorithm — a generic artist walk (v4)

v3 hand-wrote an extractor per plot type, so anything unsupported was *silent*
data loss. v4 instead walks matplotlib's own artist tree:

1. For each artist, loop its zero-argument `get_*` methods (`Artist.properties()`)
   and keep those that have a matching `set_*` — that pair is the round-trip contract.
2. Record a **structural category** (`line`/`patch`/`collection`/`image`/`text`),
   not the class name, so unknown subclasses still dispatch correctly.
3. Store arrays separately; the tree itself is plain JSON.

Two things a naive getter-loop gets wrong, both handled explicitly:

- **auto-vs-explicit state.** Replaying `set_xlim(get_xlim())` silently kills
  autoscale (MATLAB stores `XLimMode` for exactly this reason). Worse, setting
  limits on a *twin* re-clears it, so the flag is restored in a final pass.
- **construction ≠ properties.** Getters give values, not how to build the object.
  Scatter needs its identity+offset transform pair or markers render as giant
  blobs; filled contours must keep `Path` codes or their holes fill in.

The file is a zip of `tree.json` + `arrays.npz` — **no pickle anywhere**, so it
loads with `allow_pickle=False` and is readable from any language.

In [ ]:
figt, axL = plt.subplots()
axL.plot(x, np.sin(x), "b", label="sin")
axL.fill_between(x, np.sin(x)-0.15, np.sin(x)+0.15, color="b", alpha=0.2, label="band")
axL.set_ylabel("sin", color="b"); axL.tick_params(axis="y", labelcolor="b")
axR = axL.twinx()
axR.plot(x, np.exp(-x/3), "r", label="decay")
axR.set_ylabel("decay", color="r"); axR.tick_params(axis="y", labelcolor="r")
figt.savefig(OUT / "twin_fill.png")

fig_rebuilt = mfs.replot_scene(OUT / "twin_fill.mplscene")   # no pickle
[a.yaxis.label.get_color() for a in fig_rebuilt.axes]

## 5. Getting your data back — `read_data`

MATLAB's `.fig` stores the data but gives you no API to query it. Here the
scene tree already holds every array, so `read_data` hands back something that
**behaves like a dict of series** — no new vocabulary to learn. Pull out `x`
and `y`, then use matplotlib exactly as you normally would.

In [ ]:
d = mfs.read_data(OUT / "twin_fill.mplscene")

print("keys:", d.keys())                 # dict-like
for label, s in d.items():
    print(f"  {label:7s} kind={s.kind:5s} n={len(s)}")

s = d["sin"]
x_sig, y_sig = s.x, s.y                  # plain numpy arrays
band = d["band"]

fig5, ax5 = plt.subplots()               # ordinary matplotlib from here on
ax5.plot(x_sig, y_sig, label="sin")
ax5.fill_between(band.x, band.lo, band.hi, alpha=0.2, label="band")
ax5.legend()

d.to_csv(OUT / "twin_fill.csv")          # or d.to_dataframe()

## 6. Regular grids keep their shape

A `2x2` figure stores each panel's grid geometry, so `replot_scene` rebuilds
a `2x2` — not a `4x1` column. (An irregular `gridspec` falls back to a column;
use `loadfig` on the `.mplpkl` when you need an irregular layout pixel-perfect.)

In [ ]:
figg, axs = plt.subplots(2, 2, figsize=(7, 5))
for k, a in enumerate(axs.ravel()):
    a.plot(x, np.sin(x + k)); a.set_title(f"panel {k}")
figg.savefig(OUT / "grid.png")

figg2 = mfs.replot_scene(OUT / "grid.mplscene")
figg2.axes[0].get_subplotspec().get_geometry()[:2]   # -> (2, 2)

## 7. `ExperimentResult` — data + analysis + figure in one file

Bundle raw data, parameters and figures into a single `.expr` for archiving.
`extra_data` on a plain figure save works the same way via
`savefig_bundle(fig, path, extra_data={...})`, recovered through `loadfig`.

> **Security:** `.mplpkl` and `.expr` use `pickle` — only load files you trust.
> The `.mplscene` is plain arrays and always safe.

In [ ]:
r = mfs.ExperimentResult("ESR #42", params={"B_mT": 12.5, "power_dBm": -10})
r.store("freq_GHz", np.linspace(2.80, 2.95, 200))
r.store("signal", np.random.default_rng(0).normal(size=200))
r.attach_figure(figt)
r.save(OUT / "esr_042.expr")

r2 = mfs.ExperimentResult.load(OUT / "esr_042.expr")
r2.summary()

## What needs a real GUI (not shown headless)

- **Qt toolbar Save** (`%matplotlib qt`): dialog gains two checkboxes —
  *Also save plot data (.npz)* and *Also save editable figure (.mplpkl)* — and
  writes a `<name>/` folder. Unticking both = plain image, as before.
- **ipympl Save** (`%matplotlib widget`): `dialog` mode opens a local Python
  dialog (nothing written if you cancel); `timestamp` mode writes
  `plot_<timestamp>[_suptitle]/` into `save_dir` and lets the browser download
  proceed — the right choice over SSH.
